# 🔌 Model Integration: OpenAI, Google Gemini & GROQ

## Learning Objectives
In this notebook, you will learn:
1. **`init_chat_model`** - one provider-agnostic constructor that swaps models with a string
2. **Provider classes** - when to reach for `ChatOpenAI` / `ChatGoogleGenerativeAI` / `ChatGroq` directly instead
3. **Streaming** - render tokens as they are generated with `.stream()` for a responsive UX
4. **Batching** - run independent prompts in parallel with `.batch()` and cap concurrency

## Prerequisites
- Completed `1-langchainintro.ipynb`
- `pip install langchain langchain-openai python-dotenv`
- Optional providers: `pip install langchain-google-genai langchain-groq`
- A `.env` file with `OPENAI_API_KEY` (plus `GOOGLE_API_KEY` / `GROQ_API_KEY` for the optional sections)

---
## 🔑 Part 1: Environment Setup

Only the provider you actually call needs a key. The Gemini and GROQ sections below are left
commented out so the notebook runs top-to-bottom with an OpenAI key alone — uncomment the
matching `.env` line and the section together.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API credentials from .env
# ============================================================================
import os

from dotenv import load_dotenv

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "❌ Set OPENAI_API_KEY in your .env file"
print("✅ OpenAI key loaded")

# Uncomment alongside the matching provider section further down:
# print("✅ Google key loaded" if os.getenv("GOOGLE_API_KEY") else "⚠️ No GOOGLE_API_KEY")
# print("✅ GROQ key loaded" if os.getenv("GROQ_API_KEY") else "⚠️ No GROQ_API_KEY")

---
## 🌐 Part 2: `init_chat_model` — the Provider-Agnostic Way

`init_chat_model` builds a chat model from a **string**, inferring the provider from the model
name (or from an explicit `provider:model` prefix). This is the recommended default: switching
from OpenAI to Gemini becomes a one-string edit rather than an import change.

### Key Concepts:
- **Inferred provider**: `"gpt-4.1"` → OpenAI, because the name is unambiguous
- **Explicit provider**: `"openai:gpt-4.1"`, `"google_genai:gemini-2.5-flash"`,
  `"groq:qwen/qwen3-32b"` — always safe, and required when a name is ambiguous
- **Uniform return type**: whatever you name, you get a `BaseChatModel` with the same
  `.invoke()` / `.stream()` / `.batch()` / `.bind_tools()` surface

In [ ]:
# ============================================================================
# MODEL INIT: Provider-agnostic construction
# ============================================================================
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-4.1")

print(f"🤖 Loaded: {type(model).__name__} -> {model.model_name}")

In [ ]:
# ============================================================================
# FIRST INVOCATION: A single prompt in, one AIMessage out
# ============================================================================
response = model.invoke("Hello How are you?")
response

### 2.1 🏭 Provider Classes Directly

`ChatOpenAI` and friends are still there and still supported. Use them when you need
constructor arguments that only that provider has (custom `base_url`, organisation IDs,
provider-specific tuning). Otherwise prefer `init_chat_model` — it keeps the notebook
portable.

> **Note**: both routes produce the same object type; `init_chat_model("gpt-4.1")` simply
> constructs a `ChatOpenAI` for you under the hood.

In [ ]:
# ============================================================================
# PROVIDER CLASS: ChatOpenAI constructed explicitly
# ============================================================================
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1")
response = model.invoke("Hello How are you?")
response

In [ ]:
# ============================================================================
# READING THE RESPONSE: .content is the plain text
# ============================================================================
print("🤖", response.content)

---
## ✨ Part 3: Google Gemini Integration

Kept commented so the notebook runs without a Google key. To enable:
`pip install langchain-google-genai` and set `GOOGLE_API_KEY` in `.env`.

Note the two equivalent routes — the string form and the class form — exactly mirroring the
OpenAI pair above. That symmetry is the point of the abstraction.

In [ ]:
# ============================================================================
# GEMINI (OPTIONAL): via init_chat_model
# ============================================================================
# from langchain.chat_models import init_chat_model
#
# model = init_chat_model("google_genai:gemini-2.5-flash")
# response = model.invoke("Why do parrots talk?")
# print("🤖", response.content)

In [ ]:
# ============================================================================
# GEMINI (OPTIONAL): via the provider class
# ============================================================================
# from langchain_google_genai import ChatGoogleGenerativeAI
#
# model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
# response = model.invoke("Why do parrots talk?")
# response

---
## ⚡ Part 4: GROQ Integration

Same two routes again. GROQ serves open-weight models (Llama, Qwen, Mixtral) on custom
inference hardware, so it is the usual choice when latency matters more than frontier quality.

To enable: `pip install langchain-groq` and set `GROQ_API_KEY` in `.env`.

In [ ]:
# ============================================================================
# GROQ (OPTIONAL): via init_chat_model
# ============================================================================
# from langchain.chat_models import init_chat_model
#
# model = init_chat_model("groq:qwen/qwen3-32b")
# response = model.invoke("Why do parrots talk?")
# print("🤖", response.content)

In [ ]:
# ============================================================================
# GROQ (OPTIONAL): via the provider class
# ============================================================================
# from langchain_groq import ChatGroq
#
# model = ChatGroq(model="qwen/qwen3-32b")
# response = model.invoke("Why do parrots talk?")
# response

---
## 🌊 Part 5: Streaming

Most models can emit output while it is still being generated. `.stream()` returns an iterator
of chunks instead of one finished message, which dramatically improves perceived latency on
long responses — the user sees the first words in a few hundred milliseconds rather than
staring at a spinner for ten seconds.

### Key Concepts:
- **`.invoke()`**: blocks, returns one complete `AIMessage`
- **`.stream()`**: yields `AIMessageChunk`s as they arrive; `chunk.text` is the new text
- **Same total cost**: streaming changes *when* you see tokens, not how many you pay for

In [ ]:
# ============================================================================
# BASELINE: Blocking invoke — nothing renders until the whole answer is done
# ============================================================================
model.invoke("Write me a 200 words paragraph on Artificial Intelligence")

In [ ]:
# ============================================================================
# STREAMING: Same prompt, rendered chunk by chunk
# ============================================================================
# The "|" separator makes the chunk boundaries visible; drop it in real apps.
for chunk in model.stream("Write me a 200 words paragraph on Artificial Intelligence"):
    print(chunk.text, end="|", flush=True)

In [ ]:
# ============================================================================
# STREAMING: A second example to compare chunk sizes across prompts
# ============================================================================
for chunk in model.stream("Why do parrots have colorful feathers?"):
    print(chunk.text, end="|", flush=True)

---
## 📦 Part 6: Batching

When you have several **independent** prompts, `.batch()` sends them in parallel instead of
looping. This cuts wall-clock time roughly by the degree of parallelism and lets the provider
schedule the work efficiently.

> **Note**: batching is for prompts that do not depend on each other. Anything conversational —
> where request *n+1* needs the answer to *n* — must stay sequential.

In [ ]:
# ============================================================================
# BATCH: Three independent prompts, processed in parallel
# ============================================================================
responses = model.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?",
])

for i, response in enumerate(responses):
    print(f"\n📄 [{i}] {response.content[:200]}...")

In [ ]:
# ============================================================================
# BATCH: Capping parallelism to stay inside provider rate limits
# ============================================================================
model.batch(
    [
        "Why do parrots have colorful feathers?",
        "How do airplanes fly?",
        "What is quantum computing?",
    ],
    config={
        "max_concurrency": 5,  # at most 5 in-flight requests at a time
    },
)

---
## 📝 Summary

In this notebook, we learned:

### 1. Two Ways to Construct a Model
- **`init_chat_model("gpt-4.1")`**: provider-agnostic, swap models by editing a string
- **`ChatOpenAI(...)` / `ChatGroq(...)`**: reach for these only when you need
  provider-specific constructor arguments
- **Explicit prefixes** (`"openai:"`, `"google_genai:"`, `"groq:"`) remove all ambiguity

### 2. One Interface, Many Providers
- **Uniform surface**: every provider returns a `BaseChatModel` with the same
  `.invoke()` / `.stream()` / `.batch()` / `.bind_tools()` methods
- **Portability**: OpenAI → Gemini → GROQ is a one-line change

### 3. Streaming
- **`.stream()`** yields chunks; `chunk.text` holds the incremental text
- **UX, not cost**: same token count, far better perceived latency

### 4. Batching
- **`.batch()`** runs independent prompts in parallel
- **`config={"max_concurrency": n}`** keeps you inside provider rate limits

### Next Steps
- **`3-tools.ipynb`** — give a model tools with `@tool` and `bind_tools`, then run the
  tool-execution loop by hand